In [4]:
import torch
from torch import nn
from matplotlib import pyplot as plt
import numpy as np
import os
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# set up some print options
#np.set_printoptions(precision = 3, suppress = True)
torch.set_printoptions(precision=2,linewidth=160)
#begin loggin
cur_dir = 'synthetic' 
os.makedirs(cur_dir, exist_ok=True)

# Set up common problem parameters
lr = 0.02
clip_r = lr
n = 10        # dimension
n_head = 1  # 1-headed attention
batch_size = 10000  # 1000 minibatch size
var = 0.1  # initializations scale of transformer parameter



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.set_device(3)

In [5]:
def gen_fc_unit_incidence(n):
    d = int(n * (n-1)/2)
    B = torch.zeros(n, d).cuda()
    edge_id = 0
    for i in range(n):
        for j in range(n):
            if j <=i:
                continue
            else:
                B[i,edge_id] = 1
                B[j,edge_id] = -1
                edge_id += 1
    return B.to(device)

def get_laplacian(B):
    #i is batch index
    return torch.einsum('ijk,ilk->ijl', B, B)

def gen_csl_unit_incidence(n):
    d = 2*n
    B = torch.zeros(4, n, d).to(device)
    skip_values = [2,4,6,8]
    # loop over skip values
    for sk_i in range(len(skip_values)):
        list_src = []
        list_dst = []
        skip_value = skip_values[sk_i]
        for i in range(n):
            # cycle
            list_src.append(i)
            list_dst.append((i+1)%n)
            # skip connection
            list_src.append(i)
            list_dst.append((i+skip_value)%n)
        for (ind_e, (i,j)) in enumerate(zip(list_src, list_dst)):
            B[sk_i, i, ind_e] = 1
            B[sk_i, j, ind_e] = -1
    return B
    
def generate_B_inplace(Z, nv):
    batch_size = Z.shape[0]
    if settings['data']=='csl':
        Z[:,:,0:d] = gen_csl_unit_incidence(n)[:,:,:].repeat(int(batch_size/4),1,1)
    else:
        Z[:,:,0:d] = gen_fc_unit_incidence(n)[None,:,:].repeat(int(batch_size),1,1)
    rand_edge_weights = (torch.exp((torch.rand([batch_size, d]))*2-1))[:,None,:].to(device)
    Z[:,:,0:d] = Z[:,:,0:d] * rand_edge_weights
    Z[:,:,0:d] = Z[:,:,torch.randperm(d)]
    Z[:,0:n,:] = Z[:,torch.randperm(n),:]
    return Z
def generate_demand_inplace(Z, nv):
    batch_size = Z.shape[0]
    
    if settings['obj'] == 'ev':
        Z[:,:,d:d+n]=0
    else:
        Z[:,:,d:d+n]=torch.randn(batch_size,n,n)
    
    if settings['architecture']=='stack':
        Z[:,:,d+n:d+n+n]=torch.zeros(1,n,n).expand(batch_size,n,n)

    #if settings['obj']=='heat':
    Z[:,:,d:d+n+n]=Z[:,:,d:d+n+n] - Z[:,:,d:d+n+n].mean(dim=1)[:,None,:]
    return Z

def get_target(Z, k=None):
    lap = get_laplacian(Z[:,:,0:d])
    if settings['obj']== 'electric':
        inv_lap = torch.linalg.pinv(lap)
        target = torch.einsum( 'BNk, BNM->BMk', (Z[:,:,d:d+n], inv_lap))
    elif settings['obj'] == 'resist':
        lamb, eigv = torch.linalg.eigh(lap)
        lamb[:,0].fill_(1)
        eigv[:,:,0].zero_()
        invsqrt = torch.einsum('BNi,Bi,BMi->BNM',(eigv,lamb**(-0.5),eigv))
        target = torch.einsum( 'BNk, BNM->BMk', (Z[:,:,d:d+n], invsqrt))
    elif settings['obj']=='heat':
        lamb, eigv = torch.linalg.eigh(lap)
        lamb[:,0].fill_(1)
        eigv[:,:,0].zero_()
        heat_mat = torch.einsum('BNi,Bi,BMi->BNM',(eigv,torch.exp(-temp*lamb),eigv))
        target = torch.einsum( 'BNk, BNM->BMk', (Z[:,:,d:d+n], heat_mat))
    elif settings['obj']=='ev':
        _, target  = torch.linalg.eigh(lap)
        target = target.real
    return target

def get_loss(output, target, inds=None):
    if settings['obj'] in ['electric', 'resist', 'heat']:
        if settings['architecture']=='stack':
            predicted_inverse_laplacian = output[:,:,d+n:d+n+n]
        else:
            predicted_inverse_laplacian = output[:,:,d:d+n]
        predicted_inverse_laplacian = predicted_inverse_laplacian/(1e-7 + predicted_inverse_laplacian.norm(p=2,dim=[1,2])[:,None,None])
        target = target/target.norm(p=2,dim=[1,2])[:,None,None]
        loss = ((predicted_inverse_laplacian - target).norm(p=2, dim=[1,2])**2).mean()
    else:
        assert False
    return loss

In [6]:
def linear_attention(P,Q,K,Z):
    n = Z.shape[1]
    QK = torch.einsum('Hji,Hjk->Hik', (Q,K))
    Attn = torch.einsum('BNi, Hij, BMj -> HBNM', (Z,QK,Z))
    PZ = torch.einsum('Hij, BNj -> HBNi', (P,Z))
    Output = torch.einsum('HBNi, HBNM -> BMi', (PZ,Attn))
    return Output /n
    
class Transformer_LG(nn.Module):
    def __init__(self, n_layer, n_head, n, d, var):
        super(Transformer_LG, self).__init__()
        # dnchange
        if settings['architecture'] == 'stack':
            self.register_parameter('allparam', torch.nn.Parameter(torch.zeros(n_layer, n_head, 5, d+n+n, d+n+n)))
        else:
            self.register_parameter('allparam', torch.nn.Parameter(torch.zeros(n_layer, n_head, 5, d+n, d+n)))
        with torch.no_grad():
            self.allparam.normal_(0,var)
            self.allparam[:,:,0,:,:].zero_()
            if settings['architecture'] == 'stack':
                self.allparam[:,:,1,:,:].copy_(torch.eye(d+n+n).to(device).expand_as(self.allparam[:,:,1,:,:]))
                self.allparam[:,:,2,:,:].copy_(torch.eye(d+n+n).to(device).expand_as(self.allparam[:,:,2,:,:]))
            else:
                self.allparam[:,:,1,:,:].copy_(torch.eye(d+n).to(device).expand_as(self.allparam[:,:,1,:,:]))
                self.allparam[:,:,2,:,:].copy_(torch.eye(d+n).to(device).expand_as(self.allparam[:,:,2,:,:]))
            #self.allparam[:,:,3,:,:].copy_(torch.eye(d+n+n).to(device).expand_as(self.allparam[:,:,3,:,:]))
            self.allparam[:,:,3,:,:].zero_()
            self.allparam.normal_(0,0.1)
        self.n_layer = n_layer
        self.n_head = n_head
        self.n = n
        self.d = d

    def forward(self, Z, early_stop = 1000):
        d = self.d
        n = self.n
        if settings['obj'] == 'ev':
            if settings['architecture'] == 'stack':
                init_mask = torch.cat( (torch.zeros(n,d).to(device), self.allparam[0,0,4,0:n,0:n+n]), dim = 1)
            else:
                init_mask = torch.cat( (torch.zeros(n,d).to(device), self.allparam[0,0,4,0:n,0:n]), dim = 1)
            Z = Z + init_mask[None,:,:]

        P = self.allparam[:,:,0,:,:]
        Q = self.allparam[:,:,1,:,:]
        K = self.allparam[:,:,2,:,:]
        S = self.allparam[:,0,3,:,:]
        if settings['PB']=='zero':
            if settings['architecture']=='stack':
                Qmask = torch.ones(d+n+n).to(device)
                Qmask[d:d+n] = 0
                Q = Q * Qmask[None,None,:,None]
                Q = Q * Qmask[None,None,None,:]
                K = K * Qmask[None,None,:,None]
                K = K * Qmask[None,None,None,:]
                
                Pmask = torch.ones(d+n+n).to(device)
                Pmask[d:d+n] = 0
                P = P * Pmask[None,None,:,None]
                P = P * Pmask[None,None,None,:]
    
                Smask = torch.zeros(d+n+n,d+n+n).to(device)
                Smask[d+n:d+n+n, d:d+n]=1
                S = S * Smask[None,:,:]
            else:
                assert False

        for i in range(self.n_layer):
            Zi = Z
            residues = 0
            # the forwarad map of each layer is given by F(Z) = Z + attention(Z)
            Pij = P[i,:,:,:]
            Qij = Q[i,:,:,:]
            Kij = K[i,:,:,:]
            Sij = S[i,:,:]
                
            
            residues = residues + linear_attention(Pij,Qij,Kij,Zi)
            
            if settings['architecture']=='stack':
                if settings['ff']=='no':
                    Z = Zi + residues 
                else:
                    Z = Zi + residues + torch.nn.functional.linear(input = Z, weight = Sij)
                norms = torch.cat(((Z[:,:,0:d].norm(p=2,dim=[1,2]))[:,None,None].expand(Z.shape[0],n,d),
                  (Z[:,:,d:d+n+n]+1e-6).norm(p=2,dim=[1,2])[:,None,None].expand(Z.shape[0],n,n+n)), dim=2)
                Z = Z/(norms[:,:,:])
            else:
                Z = Zi + residues + torch.nn.functional.linear(input = Z, weight = Sij)
                Z = Z/Z.norm(p=2,dim=[1,2])[:,None,None]
            if early_stop == i:
                break
        return Z
        
# a convenience function for taking a step and clipping
def clip_and_step(allparam, optimizer, clip_r = None):
    norm_p=None
    grad_all = allparam.grad
    norm_p = grad_all.norm().item()
    if norm_p > clip_r:
        grad_all.mul_(clip_r/norm_p)
        fraction = clip_r/norm_p
    else:
        fraction = 1.0
    optimizer.step()
    return fraction

In [ ]:
seeds=[0,1,2]
n_layers = [3,5]  # number of layers of transformer
n_heads = [1]
keys = []
for n_layer in n_layers:
    for n_head in n_heads:
        for s in seeds:
            keys.append((n_layer,n_head,s,))
clip_r=1

for PB in ['zero']:
    for obj in ['electric']:#['resist', 'electric', 'heat']:
        for data in ['csl']:
            settings={'architecture':'stack', 'obj':obj, 'data':data, 'PB':PB, 'ff':'yes'}
            iter_dict = {9:40100, 7:14100, 5:10100, 3:6100, 1:3100}
            print(f"lr: {lr}")
            
            temp=0.5
            
            if settings['data']=='csl':
                d = 2*n
            else:
                d = int(n*(n-1)/2)
            
            filename = cur_dir + '/{}_{}_{}_{}_{}_constrained'.format(settings['obj'], settings['architecture'], settings['data'], settings['PB'], settings['ff'])
            try:
                hist_dict = torch.load(filename)
                hist_dict['settings'] = settings
                torch.save(hist_dict, filename)
            except:
                torch.save({}, filename)
            
            for key in keys:
                n_layer = key[0]
                n_head = key[1]
                sd = key[2]
                max_iters = iter_dict[n_layer]
                stride = int(max_iters/100)
            
                lr = 0.02/n_layer
                print(key)
                prob_seed = sd
                opt_seed = sd
                hist_list = []
                
                #set seed and initialize model
                torch.manual_seed(opt_seed)
                model = Transformer_LG(n_layer, n_head, n, d, 0.1).to(device)
                #initialize algorithm.
                optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.99), weight_decay = 0.01)
            
                # set seed
                # sample random rotation matrix
                # initialize initial training batch
                np.random.seed(prob_seed)
                torch.manual_seed(prob_seed)
                # dnchange 
                if settings['architecture'] == 'stack':
                    Z = torch.zeros([batch_size,n,d+n+n]).to(device)
                else:
                    Z = torch.zeros([batch_size,n,d+n]).to(device)
                generate_B_inplace(Z, n)
                generate_demand_inplace(Z, n)
                
                residual_lr = lr
                
                for t in range(max_iters):
                    optimizer.param_groups[0]['lr'] = residual_lr
                    if t%(20*stride)==0 and t>1:
                        residual_lr = residual_lr * 0.5
                    # save model parameters
                    if t%stride ==0:
                        hist_list.append(model.allparam.clone().detach().cpu())
            
                    target = get_target(Z)
                    output = model(Z)
                    loss = get_loss(output, target)
                    
                    # compute gradient, take step
                    loss.backward()
                    fraction = clip_and_step(model.allparam, optimizer, clip_r=clip_r)
                    norms = model.allparam.grad.norm().item()
                    #model.zero_QK_L()
                    optimizer.zero_grad()
            
                    if t%10==0:
                        generate_B_inplace(Z, n)
                        generate_demand_inplace(Z, n)
                    if t%200 ==0 or t<5:
                        print('iter {} | Loss: {:.3}, naive: {:.3}, gradnorm: {:.2}, fraction:{}'\
                              .format(t,loss.item(), 0.0, norms, fraction))
                hist_dict = torch.load(filename, map_location='cpu')
                hist_dict[key] = hist_list
                torch.save(hist_dict, filename)

#powl code below

In [175]:
def gen_fc_unit_incidence(n):
    d = int(n * (n-1)/2)
    B = torch.zeros(n, d).cuda()
    edge_id = 0
    for i in range(n):
        for j in range(n):
            if j <=i:
                continue
            else:
                B[i,edge_id] = 1
                B[j,edge_id] = -1
                edge_id += 1
    return B.to(device)

def get_laplacian(B):
    #i is batch index
    return torch.einsum('ijk,ilk->ijl', B, B)

def gen_csl_unit_incidence(n):
    d = 2*n
    B = torch.zeros(4, n, d).to(device)
    skip_values = [2,4,6,8]
    # loop over skip values
    for sk_i in range(len(skip_values)):
        list_src = []
        list_dst = []
        skip_value = skip_values[sk_i]
        for i in range(n):
            # cycle
            list_src.append(i)
            list_dst.append((i+1)%n)
            # skip connection
            list_src.append(i)
            list_dst.append((i+skip_value)%n)
        for (ind_e, (i,j)) in enumerate(zip(list_src, list_dst)):
            B[sk_i, i, ind_e] = 1
            B[sk_i, j, ind_e] = -1
    return B
    
def generate_B_inplace(Z, nv):
    batch_size = Z.shape[0]
    if settings['data']=='csl':
        Z[:,:,0:d] = gen_csl_unit_incidence(n)[:,:,:].repeat(int(batch_size/4),1,1)
    else:
        Z[:,:,0:d] = gen_fc_unit_incidence(n)[None,:,:].repeat(int(batch_size),1,1)
    rand_edge_weights = (torch.exp((torch.rand([batch_size, d]))*2-1))[:,None,:].to(device)
    Z[:,:,0:d] = Z[:,:,0:d] * rand_edge_weights
    Z[:,:,0:d] = Z[:,:,torch.randperm(d)]
    Z[:,0:n,:] = Z[:,torch.randperm(n),:]
    return Z
def generate_demand_inplace(Z, nv):
    batch_size = Z.shape[0]
    
    if settings['obj'] == 'ev':
        Z[:,:,d:d+n]=0
    else:
        Z[:,:,d:d+n]=torch.randn(batch_size,n,n)
    if settings['obj']=='heat':
        Z[:,:,d:d+n]=Z[:,:,d:d+n] - Z[:,:,d:d+n].mean(dim=1)[:,None,:]
    if settings['architecture']=='stack':
        Z[:,:,d+n:d+n+n]=torch.zeros(1,n,n).expand(batch_size,n,n)
    return Z

def get_target(Z, k=None):
    lap = get_laplacian(Z[:,:,0:d])
    if settings['obj']== 'electric':
        inv_lap = torch.linalg.pinv(lap)
        target = torch.einsum( 'BNk, BNM->BMk', (Z[:,:,d:d+n], inv_lap))
    elif settings['obj'] == 'resist':
        lamb, eigv = torch.linalg.eigh(lap)
        lamb[:,0].fill_(1)
        eigv[:,:,0].zero_()
        invsqrt = torch.einsum('BNi,Bi,BMi->BNM',(eigv,lamb**(-0.5),eigv))
        target = torch.einsum( 'BNk, BNM->BMk', (Z[:,:,d:d+n], invsqrt))
    elif settings['obj']=='heat':
        lamb, eigv = torch.linalg.eigh(lap)
        lamb[:,0].fill_(1)
        eigv[:,:,0].zero_()
        heat_mat = torch.einsum('BNi,Bi,BMi->BNM',(eigv,torch.exp(-temp*lamb),eigv))
        target = torch.einsum( 'BNk, BNM->BMk', (Z[:,:,d:d+n], heat_mat))
    elif settings['obj']=='ev':
        _, target  = torch.linalg.eigh(lap)
        target = target.real
    return target

def get_loss(output, target, inds=None):
    if settings['obj'] in ['electric', 'resist', 'heat']:
        if settings['architecture']=='stack':
            predicted_inverse_laplacian = output[:,:,d+n:d+n+n]
        elif settings['architecture']=='powl':
            predicted_inverse_laplacian = output[:,:,:]
        else:
            predicted_inverse_laplacian = output[:,:,d:d+n]
        predicted_inverse_laplacian = predicted_inverse_laplacian/(1e-7 + predicted_inverse_laplacian.norm(p=2,dim=[1,2])[:,None,None])
        target = target/target.norm(p=2,dim=[1,2])[:,None,None]
        loss = ((predicted_inverse_laplacian - target).norm(p=2, dim=[1,2])**2).mean()
    else:
        assert False
    return loss



In [221]:
def linear_attention(P,Q,K,Z):
    n = Z.shape[1]
    QK = torch.einsum('Hji,Hjk->Hik', (Q,K))
    Attn = torch.einsum('BNi, Hij, BMj -> HBNM', (Z,QK,Z))
    PZ = torch.einsum('Hij, BNj -> HBNi', (P,Z))
    Output = torch.einsum('HBNi, HBNM -> BMi', (PZ,Attn))
    return Output /n
    
class Transformer_LG(nn.Module):
    def __init__(self, n_layer, n_head, n, d, var):
        super(Transformer_LG, self).__init__()
        # dnchange
        if settings['architecture'] == 'stack':
            self.register_parameter('allparam', torch.nn.Parameter(torch.zeros(n_layer, n_head, 5, d+n+n, d+n+n)))
        else:
            self.register_parameter('allparam', torch.nn.Parameter(torch.zeros(n_layer, n_head, 5, d+n, d+n)))
        with torch.no_grad():
            self.allparam.normal_(0,var)
            self.allparam[:,:,0,:,:].zero_()
            if settings['architecture'] == 'stack':
                self.allparam[:,:,1,:,:].copy_(torch.eye(d+n+n).to(device).expand_as(self.allparam[:,:,1,:,:]))
                self.allparam[:,:,2,:,:].copy_(torch.eye(d+n+n).to(device).expand_as(self.allparam[:,:,2,:,:]))
            else:
                self.allparam[:,:,1,:,:].copy_(torch.eye(d+n).to(device).expand_as(self.allparam[:,:,1,:,:]))
                self.allparam[:,:,2,:,:].copy_(torch.eye(d+n).to(device).expand_as(self.allparam[:,:,2,:,:]))
            #self.allparam[:,:,3,:,:].copy_(torch.eye(d+n+n).to(device).expand_as(self.allparam[:,:,3,:,:]))
            self.allparam[:,:,3,:,:].zero_()
        self.n_layer = n_layer
        self.n_head = n_head
        self.n = n
        self.d = d

    def forward(self, Z, early_stop = 1000):
        d = self.d
        n = self.n
        if settings['obj'] == 'ev':
            if settings['architecture'] == 'stack':
                init_mask = torch.cat( (torch.zeros(n,d).to(device), self.allparam[0,0,4,0:n,0:n+n]), dim = 1)
            else:
                init_mask = torch.cat( (torch.zeros(n,d).to(device), self.allparam[0,0,4,0:n,0:n]), dim = 1)
            Z = Z + init_mask[None,:,:]
            
        for i in range(self.n_layer):
            Zi = Z
            residues = 0
            # the forwarad map of each layer is given by F(Z) = Z + attention(Z)
            Pij = self.allparam[i,:,0,:,:]
            Qij = self.allparam[i,:,1,:,:]
            Kij = self.allparam[i,:,2,:,:]
            Sij = self.allparam[i,0,3,:,:]

            if settings['PB']=='zero':
                if settings['architecture']=='stack':
                    Pmask = torch.zeros(d+n+n).to(device)
                    Pmask[d:d+n+n] = 1
                else:
                    Pmask = torch.zeros(d+n).to(device)
                    Pmask[d:d+n] = 1
                Pij = Pij * Pmask[None,:,None]
                Sij = Sij * Pmask[:,None]
            
            residues = residues + linear_attention(Pij,Qij,Kij,Zi)
            
            if settings['architecture']=='stack':
                if settings['ff']=='no':
                    Z = Zi + residues 
                else:
                    Z = Zi + residues + torch.nn.functional.linear(input = Z, weight = Sij)
                norms = torch.cat(((Z[:,:,0:d].norm(p=2,dim=[1,2]))[:,None,None].expand(Z.shape[0],n,d),
                  (Z[:,:,d:d+n+n]+1e-6).norm(p=2,dim=[1,2])[:,None,None].expand(Z.shape[0],n,n+n)), dim=2)
                Z = Z/(norms[:,:,:])
            else:
                Z = Zi + residues + torch.nn.functional.linear(input = Z, weight = Sij)
                Z = Z/Z.norm(p=2,dim=[1,2])[:,None,None]
            if early_stop == i:
                break
        return Z
        
# a convenience function for taking a step and clipping
def clip_and_step(allparam, optimizer, clip_r = None):
    norm_p=None
    grad_all = allparam.grad
    norm_p = grad_all.norm().item()
    if norm_p > clip_r:
        grad_all.mul_(clip_r/norm_p)
        fraction = clip_r/norm_p
    else:
        fraction = 1.0
    optimizer.step()
    return fraction

class PowerOfMat(nn.Module):
    def __init__(self, n_layer, n):
        super(PowerOfMat, self).__init__()
        self.register_parameter('allparam', torch.nn.Parameter(torch.zeros(n_layer, n, n)))
        with torch.no_grad():
            for i in range(n_layer):
                self.allparam[i,:,:].data.copy_(torch.eye(n).to(device))
    def forward(self, B, chi):
        out = self.allparam[0,:,:]
        L0 = torch.einsum('BNi, BMi->BNM',(B,B))
        L = L0
        #L = L / L.norm(p=2, dim=[1]).mean(1)[:,None,None]
        for i in range(1,n_layer):
            L = L / L.norm(p=2, dim=[1,2])[:,None,None]
            out = out + torch.nn.functional.linear(L, self.allparam[i,:,:])
            
            L = torch.einsum('Bij,Bjk->Bik', L, L0)
            
        out = out/ out.norm(p=2, dim=[1,2])[:,None,None]
        out = torch.einsum('BNi, BNM-> BMi', (chi, out))
        
        return out
        

In [222]:
seeds=[1,0,2]
n_layers = [11,9,7,5,3]  # number of layers of transformer
n_heads = [1]
keys = []
for n_layer in n_layers:
    for n_head in n_heads:
        for s in seeds:
            keys.append((n_layer,n_head,s,))
clip_r=1

for PB in ['all']:
    for obj in ['electric','resist','head']:
        for data in ['csl', 'fc']:
            settings={'architecture':'powl', 'obj':obj, 'data':data, 'PB':PB, 'norm':'yes'}
            iter_dict = {11: 20100, 13: 20100, 9:20100, 7:10100, 5:10100, 3:10100, 2:3100, 1:3100}
            print(f"lr: {lr}")
            
            temp=0.5
            
            if settings['data']=='csl':
                d = 2*n
            else:
                d = int(n*(n-1)/2)
            
            filename = cur_dir + '/{}_{}_{}_{}_{}'.format(settings['obj'], settings['architecture'], settings['data'], settings['PB'])
            try:
                hist_dict = torch.load(filename)
                hist_dict['settings'] = settings
                torch.save(hist_dict, filename)
            except:
                torch.save({}, filename)
            
            for key in keys:
                n_layer = key[0]
                n_head = key[1]
                sd = key[2]
                max_iters = iter_dict[n_layer]
                stride = int(max_iters/100)

                if sd==1 and data == 'csl':
                    continue
            
                lr = 0.02/n_layer
                print(key)
                prob_seed = sd
                opt_seed = sd
                hist_list = []
                
                #set seed and initialize model
                torch.manual_seed(opt_seed)
                model = PowerOfMat(n_layer, n).to(device)
                #initialize algorithm.
                optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.99), weight_decay = 0.001)
            
                # set seed
                # initialize initial training batch
                np.random.seed(prob_seed)
                torch.manual_seed(prob_seed)
                # dnchange 
                if settings['architecture'] == 'stack':
                    Z = torch.zeros([batch_size,n,d+n+n]).to(device)
                else:
                    Z = torch.zeros([batch_size,n,d+n]).to(device)
                generate_B_inplace(Z, n)
                generate_demand_inplace(Z, n)
                
                residual_lr = lr
                
                for t in range(max_iters):
                    optimizer.param_groups[0]['lr'] = residual_lr
                    if t%(5*stride)==0 and t>1:
                        residual_lr = residual_lr * 0.5
                    # save model parameters
                    if t%stride ==0:
                        hist_list.append(model.allparam.clone().detach().cpu())
            
                    target = get_target(Z)
                    output = model(Z[:,:,0:d], Z[:,:,d:d+n])
                    loss = get_loss(output, target)
                    
                    # compute gradient, take step
                    loss.backward()
                    fraction = clip_and_step(model.allparam, optimizer, clip_r=clip_r)
                    norms = model.allparam.grad.norm().item()
                    #model.zero_QK_L()
                    optimizer.zero_grad()
            
                    if t%10==0:
                        generate_B_inplace(Z, n)
                        generate_demand_inplace(Z, n)
                    if t%200 ==0 or t<5:
                        print('iter {} | Loss: {:.3}, naive: {:.3}, gradnorm: {:.2}, fraction:{}'\
                              .format(t,loss.item(), 0.0, norms, fraction))
                hist_dict = torch.load(filename, map_location='cpu')
                hist_dict[key] = hist_list
                torch.save(hist_dict, filename)

lr: 0.0022222222222222222
(11, 1, 0)
iter 0 | Loss: 1.37, naive: 0.0, gradnorm: 0.12, fraction:1.0
iter 1 | Loss: 1.37, naive: 0.0, gradnorm: 0.12, fraction:1.0
iter 2 | Loss: 1.37, naive: 0.0, gradnorm: 0.12, fraction:1.0
iter 3 | Loss: 1.37, naive: 0.0, gradnorm: 0.12, fraction:1.0
iter 4 | Loss: 1.36, naive: 0.0, gradnorm: 0.12, fraction:1.0
iter 200 | Loss: 0.942, naive: 0.0, gradnorm: 0.15, fraction:1.0
iter 400 | Loss: 0.363, naive: 0.0, gradnorm: 0.097, fraction:1.0
iter 600 | Loss: 0.287, naive: 0.0, gradnorm: 0.077, fraction:1.0
iter 800 | Loss: 0.188, naive: 0.0, gradnorm: 0.084, fraction:1.0
iter 1000 | Loss: 0.153, naive: 0.0, gradnorm: 0.08, fraction:1.0
iter 1200 | Loss: 0.133, naive: 0.0, gradnorm: 0.088, fraction:1.0
iter 1400 | Loss: 0.11, naive: 0.0, gradnorm: 0.08, fraction:1.0
iter 1600 | Loss: 0.0899, naive: 0.0, gradnorm: 0.072, fraction:1.0
iter 1800 | Loss: 0.0832, naive: 0.0, gradnorm: 0.082, fraction:1.0
iter 2000 | Loss: 0.0807, naive: 0.0, gradnorm: 0.077, f